# Relative-pose Flow Matching (`fm_rel_pose`)

1. Re-evaluate the existing `fm_abs_pose` checkpoints from Drive with metrics **m2** (no retraining).
2. Train `fm_rel_pose` with the identical architecture, data and budget.
3. Paired comparison on the same held-out conditions, K and seeds.

Pre-registered primary contrast for H1: **E1a `eval_ood_position`** (mode calibration, conditional dispersion, endpoint). In-distribution and E1b are secondary. This compares *frames only*; do not read it as a geometry effect.

Runtime: GPU (T4). Expected ≈ 35–45 min, mostly the two training runs (~13.5 min each on T4).

## 1. Code + Drive

In [ ]:
import os, sys, glob, json, subprocess
REPO_URL = ""  # Optional Git remote URL for Colab; otherwise upload the repository ZIP.
IN_COLAB = "google.colab" in sys.modules

if not os.path.isdir("camera_motion"):
    candidates = sorted(os.path.dirname(p) for p in glob.glob("*/camera_motion") if os.path.isdir(p))
    if not candidates and IN_COLAB:
        if REPO_URL:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "camera-motion-flow-matching"], check=True)
        else:
            from google.colab import files
            from zipfile import ZipFile
            uploaded = files.upload()
            if not uploaded:
                raise RuntimeError("Upload the repository ZIP or specify REPO_URL.")
            with ZipFile(next(iter(uploaded))) as zf:
                zf.extractall(".")
        candidates = sorted(os.path.dirname(p) for p in glob.glob("*/camera_motion") if os.path.isdir(p))
    if candidates:
        os.chdir(candidates[0])
assert os.path.isdir("camera_motion"), f"Cannot find repository root: {os.getcwd()}"
sys.path.insert(0, os.getcwd())
print("repository root:", os.getcwd())


RUNS = "runs"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = "/content/drive/MyDrive/camera_motion_runs"
SEED = 0
ABS = {f: f"{RUNS}/fm_abs_pose_{f}_s{SEED}" for f in ("E1a", "E1b")}
REL = {f: f"{RUNS}/fm_rel_pose_{f}_s{SEED}" for f in ("E1a", "E1b")}
for f, d in ABS.items():
    assert os.path.exists(f"{d}/ckpt_last.pt"), f"missing abs_pose checkpoint for {f}: {d}"
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("abs checkpoints found:", list(ABS.values()))

## 2. Tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], capture_output=True, text=True)
print(r.stdout[-600:]); assert r.returncode == 0, r.stderr[-1500:]

## 3. Re-evaluate `fm_abs_pose` with metrics m2
Primary: Euler NFE 50 (same as gate 2). Diagnostic: midpoint NFE 100 on E1a — if jerk drops a lot, the excess jerk is solver error rather than a learned property.

In [ ]:
!python -u scripts/eval_fm.py --run {ABS['E1a']} --K 200 --nfe 50
!python -u scripts/eval_fm.py --run {ABS['E1b']} --K 200 --nfe 50
!python -u scripts/eval_fm.py --run {ABS['E1a']} --K 200 --nfe 100 --method midpoint

## 4. Train `fm_rel_pose` (identical budget). Re-running resumes.

In [ ]:
!python -u scripts/train_fm.py --rep rel_pose --family E1a --n-train 10000 --steps 20000 --batch 256 --seed {SEED} --log-every 1000 --out {REL['E1a']}

In [ ]:
!python -u scripts/train_fm.py --rep rel_pose --family E1b --n-train 10000 --steps 20000 --batch 256 --seed {SEED} --log-every 1000 --out {REL['E1b']}

## 5. Evaluate `fm_rel_pose`

In [ ]:
!python -u scripts/eval_fm.py --run {REL['E1a']} --K 200 --nfe 50
!python -u scripts/eval_fm.py --run {REL['E1b']} --K 200 --nfe 50

## 6. Paired comparison (b = rel_pose minus a = abs_pose)
`*` = paired 95% CI excludes 0. Negative is better for every error metric.

In [ ]:
EV = "eval_m2_K200_nfe50_euler"
os.makedirs(f"{RUNS}/comparisons", exist_ok=True)
for f in ("E1a", "E1b"):
    !python -u scripts/compare_runs.py --a {ABS[f]}/{EV} --b {REL[f]}/{EV} --out {RUNS}/comparisons/abs_vs_rel_{f}.json

## 7. Figures

In [ ]:
from IPython.display import Image, display
for name, d in [("abs E1a", ABS["E1a"]), ("rel E1a", REL["E1a"]), ("abs E1b", ABS["E1b"]), ("rel E1b", REL["E1b"])]:
    for f in sorted(glob.glob(f"{d}/eval_m2_K200_nfe50_euler/figures/*.png")):
        print(name, os.path.basename(f)); display(Image(filename=f))

## 8. Pack results (no checkpoints)

In [ ]:
import zipfile
with zipfile.ZipFile("gate3_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for d in list(ABS.values()) + list(REL.values()) + [f"{RUNS}/comparisons"]:
        for f in glob.glob(f"{d}/**/*", recursive=True):
            if os.path.isdir(f) or f.endswith(".pt"):
                continue
            if "/eval_K200_nfe50_euler/" in f:      # gate-2 (m1) evals were already sent
                continue
            z.write(f, os.path.relpath(f, RUNS))
print("gate3_results.zip", round(os.path.getsize("gate3_results.zip") / 2**20, 1), "MB")
if IN_COLAB:
    from google.colab import files; files.download("gate3_results.zip")